# IP Geolocation API — server hosting location

In [1]:
import requests
import socket
import pandas as pd
import time

In [7]:
def get_geolocation(ip):
    """Calls ip-api.com with an IP address, returns the fields relevant
    to hosting/jurisdiction analysis."""
    if ip is None:
        return {'country': None, 'region': None, 'isp': None}
    
    fields = 'status,country,countryCode,regionName,isp,org,as,hosting,lat,lon'
    url = f'http://ip-api.com/json/{ip}?fields={fields}'
    
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        
        if data.get('status') == 'success':
            return {
                'country': data.get('country'),
                'country_code': data.get('countryCode'),
                'region': data.get('regionName'),
                'isp': data.get('isp'),
                'org': data.get('org'),
                'asn': data.get('as'),           # e.g. "AS16509 Amazon.com, Inc."
                'is_hosting': data.get('hosting'),  # True = cloud/hosting provider, not a residential ISP
                'lat': data.get('lat'),
                'lon': data.get('lon'),
            }
        else:
            return {'country': None, 'region': None, 'isp': None}
    except Exception as e:
        print(f'Error for IP {ip}: {e}')
        return {'country': None, 'region': None, 'isp': None}

let's test it  with lemonde.fr

In [8]:
test_result = get_server_location('lemonde.fr')
print(test_result)

{'country': 'France', 'country_code': 'FR', 'region': 'Île-de-France', 'isp': 'Fastly, Inc.', 'org': 'Fastly, Inc.', 'asn': 'AS54113 Fastly, Inc.', 'is_hosting': True, 'lat': 48.8558, 'lon': 2.3494, 'ip': '151.101.122.137', 'domain': 'lemonde.fr'}


testing on a bigger batch

In [9]:
test_domains = ['lemonde.fr', 'amazon.fr', 'service-public.fr', 'bbc.co.uk', 'google.com']

test_results = []
for d in test_domains:
    result = get_server_location(d)
    test_results.append(result)
    print(result)
    time.sleep(1.5)  # stay under 45 requests/minute limit

{'country': 'France', 'country_code': 'FR', 'region': 'Île-de-France', 'isp': 'Fastly, Inc.', 'org': 'Fastly, Inc.', 'asn': 'AS54113 Fastly, Inc.', 'is_hosting': True, 'lat': 48.8558, 'lon': 2.3494, 'ip': '151.101.122.137', 'domain': 'lemonde.fr'}
{'country': 'Ireland', 'country_code': 'IE', 'region': 'Leinster', 'isp': 'Amazon Technologies Inc.', 'org': 'Amazon Technologies Inc. (eu-west-1)', 'asn': 'AS16509 Amazon.com, Inc.', 'is_hosting': True, 'lat': 53.3498, 'lon': -6.26031, 'ip': '3.254.236.66', 'domain': 'amazon.fr'}
{'country': 'France', 'country_code': 'FR', 'region': 'Île-de-France', 'isp': 'Worldline France hosting', 'org': 'Worldline SA', 'asn': 'AS47957 Worldline IGSA SA', 'is_hosting': False, 'lat': 48.8575, 'lon': 2.35138, 'ip': '160.92.168.33', 'domain': 'service-public.fr'}
{'country': 'Canada', 'country_code': 'CA', 'region': 'Quebec', 'isp': 'Fastly, Inc.', 'org': 'Fastly, Inc.', 'asn': 'AS54113 Fastly, Inc.', 'is_hosting': True, 'lat': 45.5019, 'lon': -73.5674, 'ip'

let's run  it on the full dataset

In [10]:
df_urls = pd.read_csv('../clean_data/urls_new.csv')
domains = df_urls['domain'].tolist()

print(f'Total domains to process: {len(domains)}')
print(f'Estimated time: ~{round(len(domains) * 1.5 / 60, 1)} minutes')

Total domains to process: 5000
Estimated time: ~125.0 minutes


In [11]:
all_geo_results = []

for i, domain in enumerate(domains):
    result = get_server_location(domain)
    all_geo_results.append(result)
    
    if (i + 1) % 50 == 0:
        print(f'Processed {i+1}/{len(domains)}')
    
    time.sleep(1.5)  # respect rate limit

print(f'\nDone — {len(all_geo_results)} domains processed')

Processed 50/5000
Processed 100/5000
Processed 150/5000
Processed 200/5000
Processed 250/5000
Processed 300/5000
Processed 350/5000
Processed 400/5000
Processed 450/5000
Processed 500/5000
Processed 550/5000
Processed 600/5000
Processed 650/5000
Processed 700/5000
Processed 750/5000
Processed 800/5000
Processed 850/5000
Processed 900/5000
Processed 950/5000
Processed 1000/5000
Processed 1050/5000
Error for IP 45.12.50.175: HTTPConnectionPool(host='ip-api.com', port=80): Max retries exceeded with url: /json/45.12.50.175?fields=status,country,countryCode,regionName,isp,org,as,hosting,lat,lon (Caused by ConnectTimeoutError(<HTTPConnection(host='ip-api.com', port=80) at 0x12f211590>, 'Connection to ip-api.com timed out. (connect timeout=5)'))
Error for IP 141.95.185.96: HTTPConnectionPool(host='ip-api.com', port=80): Read timed out. (read timeout=5)
Error for IP 23.192.118.25: HTTPConnectionPool(host='ip-api.com', port=80): Max retries exceeded with url: /json/23.192.118.25?fields=status,c

In [12]:
df_geo = pd.DataFrame(all_geo_results)

In [13]:
df_geo.shape

(5000, 11)

In [14]:
df_geo.head()

,country,country_code,region,isp,org,asn,is_hosting,lat,lon,ip,domain
0,Germany,DE,Hesse,Amazon Technologies Inc.,AWS EC2 (eu-central-1),"AS16509 Amazon.com, Inc.",True,50.1109,8.68213,52.29.103.196,telekom.de
1,None,NaN,None,None,NaN,NaN,NaN,NaN,NaN,None,iiko.it
2,None,NaN,None,None,NaN,NaN,NaN,NaN,NaN,None,nominetdns.uk
3,Ireland,IE,Leinster,Amazon Technologies Inc.,AWS EC2 (eu-west-1),"AS16509 Amazon.com, Inc.",True,53.3498,-6.26031,34.246.241.220,t-online.de
4,Canada,CA,Quebec,"Fastly, Inc.","Fastly, Inc.","AS54113 Fastly, Inc.",True,45.5019,-73.56740,151.101.192.81,bbc.co.uk


In [15]:
df_geo['country'].value_counts()

country
Canada                    1068
Germany                    864
France                     791
United States              752
United Kingdom             221
Ireland                    145
Italy                      140
China                       62
The Netherlands             56
Netherlands                 35
Russia                      29
Singapore                   18
Australia                   16
Japan                       16
South Korea                 14
India                       11
Spain                        9
Belgium                      9
Sweden                       9
Moldova                      8
Finland                      6
South Africa                 6
Austria                      6
Czechia                      6
Hong Kong                    6
Poland                       4
Vietnam                      3
Luxembourg                   3
Brazil                       3
Israel                       3
Latvia                       3
Turkey                       3


In [16]:
df_geo.to_csv('../raw_data/api_geolocation_results_new.csv', index=False)

In [17]:
df_urls.head()

,rank,url,domain,site,extension,last_segment,ext_type,country,region,category
0,116,https://telekom.de,telekom.de,telekom,de,de,simple,DE,European,NaN
1,143,https://iiko.it,iiko.it,iiko,it,it,simple,IT,European,NaN
2,200,https://nominetdns.uk,nominetdns.uk,nominetdns,uk,uk,simple,UK,European,NaN
3,215,https://t-online.de,t-online.de,t-online,de,de,simple,DE,European,NaN
4,221,https://bbc.co.uk,bbc.co.uk,bbc,co.uk,uk,co.XX,UK,European,NaN


In [18]:
df_merged = df_urls.merge(df_geo, on='domain', how='left')

In [19]:
df_merged.head(5)

,rank,url,domain,site,extension,last_segment,ext_type,country_x,region_x,category,country_y,country_code,region_y,isp,org,asn,is_hosting,lat,lon,ip
0,116,https://telekom.de,telekom.de,telekom,de,de,simple,DE,European,NaN,Germany,DE,Hesse,Amazon Technologies Inc.,AWS EC2 (eu-central-1),"AS16509 Amazon.com, Inc.",True,50.1109,8.68213,52.29.103.196
1,143,https://iiko.it,iiko.it,iiko,it,it,simple,IT,European,NaN,None,NaN,None,None,NaN,NaN,NaN,NaN,NaN,None
2,200,https://nominetdns.uk,nominetdns.uk,nominetdns,uk,uk,simple,UK,European,NaN,None,NaN,None,None,NaN,NaN,NaN,NaN,NaN,None
3,215,https://t-online.de,t-online.de,t-online,de,de,simple,DE,European,NaN,Ireland,IE,Leinster,Amazon Technologies Inc.,AWS EC2 (eu-west-1),"AS16509 Amazon.com, Inc.",True,53.3498,-6.26031,34.246.241.220
4,221,https://bbc.co.uk,bbc.co.uk,bbc,co.uk,uk,co.XX,UK,European,NaN,Canada,CA,Quebec,"Fastly, Inc.","Fastly, Inc.","AS54113 Fastly, Inc.",True,45.5019,-73.56740,151.101.192.81


In [20]:
# Flag mismatches: TLD says European country X, but server is hosted elsewhere
df_merged['hosting_mismatch'] = (
    (df_merged['region_x'] == 'European') &
    (df_merged['country_code'].notna()) &
    (df_merged['country_y'] != 'France') &  # adjust per-country if needed
    (~df_merged['country_code'].isin(['FR','DE','UK','NL','ES','IT','BE','CH','AT','PL',
                                        'SE','DK','FI','PT','NO','IE','LU','CZ','RO','HU']))
)

print(f'European-TLD sites actually hosted outside Europe: {df_merged["hosting_mismatch"].sum()}')
print()
print('Examples:')
df_merged[df_merged['hosting_mismatch']][['domain', 'country_x', 'country_y', 'isp']].head(10)

European-TLD sites actually hosted outside Europe: 1053

Examples:


,domain,country_x,country_y,isp
4,bbc.co.uk,UK,Canada,"Fastly, Inc."
15,independent.co.uk,UK,Canada,"Fastly, Inc."
16,service.gov.uk,UK,Canada,"Fastly, Inc."
19,google.co.uk,UK,United States,Google LLC
22,welt.de,DE,United States,"Amazon.com, Inc."
34,tagesschau.de,DE,United States,Google LLC
35,n-tv.de,DE,United States,Amazon Technologies Inc.
36,leparisien.fr,FR,Canada,"Amazon.com, Inc."
40,vinted.fr,FR,Canada,"Cloudflare, Inc."
42,rightmove.co.uk,UK,United Kingdom,Rightmove Group Limited
